# पाठ 18: क्रिप्टोग्राफिक रसीदों के साथ एआई एजेंट्स को सुरक्षित करना

## व्यावहारिक नोटबुक

यह नोटबुक चार अभ्यासों के माध्यम से चलता है:

1. **अपने पहले रसीद पर हस्ताक्षर करें** एक एजेंट टूल कॉल के लिए और इसे सत्यापित करें।
2. **रसीद में छेड़छाड़ करें** और सत्यापन विफल होते देखें।
3. **तीन-रसीद श्रृंखला बनाएं** और श्रृंखला की अखंडता की पुष्टि करें।
4. **माइक्रोसॉफ्ट एजेंट फ्रेमवर्क टूल कॉल को लपेटें** ताकि हर क्रिया एक रसीद जारी करे।

सभी क्रिप्टोग्राफिक प्रिमिटिव्स अच्छी तरह से संधारित लाइब्रेरीज से आयात किए गए हैं (`pynacl` Ed25519 के लिए, `jcs` RFC 8785 कैनोनिकल JSON के लिए, `hashlib` Python मानक लाइब्रेरी से SHA-256 के लिए)। रसीद लॉजिक स्वयं सादा पायथन है जिसे आप पढ़ और संशोधित कर सकते हैं।

सेल्स को क्रम में चलाएं। प्रत्येक अनुभाग छोटा और स्वयंपूर्ण है।


## Setup

दोनों निर्भरताओं को स्थापित करें। दोनों के पास अनुमति देने वाले लाइसेंस हैं (Apache-2.0 / MIT)।


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## सहायक उपयोगिताएँ

ये दो सहायक बेस64url एन्कोडिंग (पैडिंग के बिना) और मनमाने ऑब्जेक्ट्स के SHA-256 हैशिंग को संभालते हैं। वे नोटबुक के बाकी हिस्से को रसीद तर्क पर केंद्रित रखते हैं।


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: अपना पहला रसीद साइन करें

कल्पना करें कि हमारे एजेंट **Contoso Travel** के लिए अभी अभी सिडनी से लॉस एंजेलिस के लिए एक ग्राहक की उड़ानें देखी हैं। हम इस टूल कॉल को एक साइन की गई रसीद के रूप में रिकॉर्ड करना चाहते हैं ताकि भविष्य में कोई ऑडिटर इसे बिना हम पर भरोसा किए सत्यापित कर सके।

### Step 1.1: एक साइनिंग की जनरेट करें

उत्पादन में, एजेंट की साइनिंग की हार्डवेयर सिक्योरिटी मॉड्यूल (HSM), Azure Key Vault, या किसी समान संरक्षित स्टोर में रहेगी। इस पाठ के लिए हम मेमोरी में एक नया की जनरेट करते हैं।


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: रसीद पेलोड बनाएं

पेलोड में वह सब कुछ होता है जिसे हम चाहते हैं कि रसीद प्रमाणित करे: किसने कार्रवाई की, कौन सा टूल, किस तर्क के साथ, क्या वापस आया, किस नीति के तहत, और कब। हम तर्क और परिणाम को इनलाइन शामिल करने के बजाय उनके हैश करते हैं ताकि रसीद संवेदनशील सामग्री को लीक न करे।


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: रिसीट पर हस्ताक्षर करें और उसे संयोजित करें

तीन चरण:

1. JCS का उपयोग करके पेलोड को सामान्यीकृत करें ताकि दो कार्यान्वयन जो समान तार्किक रिसीट बनाते हैं, वे बाइट-एकरूप बाइट उत्पन्न करें।
2. सामान्यीकृत बाइट्स का SHA-256 के साथ हैश करें।
3. Ed25519 प्राइवेट की से हैश पर हस्ताक्षर करें।

फिर हस्ताक्षर मूल पेलोड के साथ संलग्न किया जाता है ताकि अंतिम रिसीट तैयार हो।


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### चरण 1.4: रसीद की सत्यापन करें

सत्यापन प्रक्रिया को उलटता है। हम हस्ताक्षर को हटाते हैं, मानक हैश को पुनः गणना करते हैं, और रसीद में सार्वजनिक कुंजी के खिलाफ हस्ताक्षर की जांच करते हैं।

यह सत्यापन करने वाला एक ऑडिटर हमसे केवल रसीद स्वयं चाहता है। कॉल करने के लिए कोई सेवा नहीं, पूछताछ के लिए कोई कुंजी निर्देशिका नहीं, कोई विश्वास आवश्यक नहीं।


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

आपको `Receipt is valid: True` देखना चाहिए। एजेंट ने अपना पहला क्रिप्टोग्राफिक रूप से हस्ताक्षरित ऑडिट रिकॉर्ड बनाया है।


## अनुभाग 2: रसीद के साथ छेड़छाड़ करें

रसीदों का पूरा उद्देश्य यह है कि वे छेड़छाड़-साबित हों। आइए इसे साबित करें।

हम रसीद के ठीक एक अक्षर को संशोधित करेंगे और सत्यापन विफल होते देखेंगे।


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### क्या अभी हुआ?

जब हमने `policy_id` बदला, तो canonical बाइट्स बदल गए। उन बाइट्स का SHA-256 हैश बदल गया। सिग्नेचर (जो मूल हैश पर था) नए हैश से मेल नहीं खाता। सत्यापन सही ढंग से `False` लौटाता है।

रेसीट के किसी भी फ़ील्ड को संशोधित करने का कोई तरीका नहीं है और फिर भी इसे सत्यापित कराना संभव नहीं है, जब तक हमलावर के पास प्राइवेट की न हो। जब तक प्राइवेट की की-वॉल्ट में हो और पब्लिक की प्रकाशित हो, छेड़छाड़ छुपाना असंभव है।

खुद आज़माएं: ऊपर के सेल में `tool_name` या `agent_id` या `timestamp` को बदलें और पुनः चलाएं। हर बदलाव एक अवैध रसीद बनाता है।


## Section 3: रसीदों को एक साथ जोड़ना

एकल रसीद एक क्रिया की सुरक्षा करती है। अधिकांश एजेंट कई क्रियाएँ करते हैं। पूरी श्रृंखला को छेड़छाड़-प्रतिरोधी बनाने के लिए, हम प्रत्येक रसीद को पिछली रसीद के हैश को नई रसीद के पेलोड में शामिल करके पिछले रसीद से जोड़ते हैं।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

अगर कोई भी रसीद हटा देता है या पुनः क्रमित करता है, तो श्रृंखला ठीक उसी बिंदु पर टूट जाती है। किसी भी बाद की रसीद का सत्यापन विफल हो जाता है क्योंकि उसका `previous_receipt_hash` अब उसके पूर्ववर्ती के वास्तविक हैश से मेल नहीं खाता।


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

अब श्रृंखला को तोड़ दें बीच वाले रसीद में छेड़छाड़ करके और पुनः सत्यापन करें। छेड़ी हुई रसीद अपनी हस्ताक्षर जांच में असफल होती है, और अगली रसीद अपनी श्रृंखला-लिंक जांच में असफल होती है (क्योंकि उसकी `previous_receipt_hash` अब संशोधित बीच वाली रसीद के हैश से मेल नहीं खाती है)।


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

रसीद 0 अभी भी सत्यापित होती है (इसमें कोई संशोधन नहीं हुआ है और इसका कोई पूर्ववर्ती जिस पर निर्भर किया जा सके, नहीं है)। रसीद 1 अपनी हस्ताक्षर जांच में विफल होती है क्योंकि हमने `tool_args_hash` को बदल दिया है। रसीद 2 अपनी श्रृंखला-लिंक जांच में विफल होती है क्योंकि इसका `previous_receipt_hash` मूल (अब संशोधित) रसीद 1 के खिलाफ गणना किया गया था।

यदि कोई हमलावर संशोधित रसीद 1 को पुनः हस्ताक्षरित भी कर दे (जो वे निजी कुंजी के बिना नहीं कर सकते), तो भी रसीद 2 में श्रृंखला-लिंक असंगति छेड़छाड़ को उजागर कर देगी। परिवर्तन को छिपाने के लिए, हमलावर को संशोधन बिंदु से आगे हर रसीद को फिर से हस्ताक्षरित करना होगा, जिसके लिए निजी कुंजी का स्वामित्व आवश्यक है।


## खंड 4: रसीद हस्ताक्षर के साथ एजेंट टूल कॉल को रैप करें

एक वास्तविक परिनियोजन में, आप नहीं चाहते कि हर एजेंट लेखक को `make_receipt` कॉल करने की याद रहे। आप चाहते हैं कि हर टूल आह्वान के लिए रसीद हस्ताक्षर स्वचालित हो।

यहाँ सबसे सरल पैटर्न है: एक रैपर क्लास जो किसी भी callable टूल फ़ंक्शन को लेता है और उसका रसीद-निर्गमण संस्करण लौटाता है। यह किसी भी एजेंट फ्रेमवर्क के लिए अनुकूल है, जिसमें Microsoft Agent Framework (`agent_framework.azure`) भी शामिल है।

यदि आपके पास Azure AI Foundry प्रोजेक्ट सेट अप नहीं है, तो नीचे का स्थानीय मॉक अभी भी इस पैटर्न को दर्शाता है।


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Framework के साथ एकीकरण

ऊपर दिया गया `ReceiptedTool` रैपर फ्रेमवर्क-रहित है। इसे Microsoft Agent Framework के साथ बनाये गए एजेंट के अंदर उपयोग करने के लिए, रैप्ड फ़ंक्शन को एक टूल के रूप में रजिस्टर करें। एक स्केच (आप मॉक को असली Azure AI Foundry टूल रजिस्ट्रेशन से बदल देंगे):

```python
# इंटीग्रेशन आकार दिखाने वाला छद्मकोड।
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="आप एक कंटोसो ट्रैवल एजेंट हैं ...",
#     tools=[receipted_lookup],   # लिपटे हुए उपकरण, कच्चे फंक्शन नहीं
# )
# response = agent.run("सिडनी से लॉस एंजेलिस के लिए जून में फ्लाइट खोजें।")
#
# # रन के बाद, प्रत्येक उपकरण कॉल जो एजेंट ने बनाया है उसके पास एक हस्ताक्षरित रसीद होती है:
# audit_chain = receipted_lookup.receipts
```

एजेंट फ्रेमवर्क को रसीदों के बारे में कुछ भी जानने की आवश्यकता नहीं है। रसीद पर हस्ताक्षर टूल के चारों ओर लिपटा हुआ है, फ्रेमवर्क में स्थिर नहीं है। इस तरह आप एजेंट को पुनः लिखे बिना मौजूदा एजेंट कोड में स्रोत जोड़ सकते हैं।


## पुनरावलोकन और बढ़ा हुआ चुनौती

आपने किया है:

- एक Ed25519 कुंजी जोड़ी उत्पन्न की है।
- एक एजेंट टूल कॉल के लिए रसीद बनाई और हस्ताक्षरित की है।
- केवल सार्वजनिक कुंजी का उपयोग करके ऑफ़लाइन रसीद की पुष्टि की है।
- एक रसीद में छेड़छाड़ की और सत्यापन विफल होता देखा है।
- तीन रसीदों की हैश-श्रृंखला बनाई है।
- श्रृंखला के मध्य में छेड़छाड़ की और दोनों, हस्ताक्षर विफलता और श्रृंखला-लिंक विफलता देखी है।
- एक उपकरण फ़ंक्शन को स्वचालित रसीद हस्ताक्षर के साथ लिपटा है।

**बढ़ा हुआ चुनौती।** रसीद स्कीमा को `request_id` फ़ील्ड (वितरित ट्रेसिंग के लिए एक UUID) के साथ बढ़ाएं। `make_receipt` को इसे शामिल करने के लिए अपडेट करें, और पुष्टि करें कि रसीदें अंत से अंत तक अभी भी सत्यापित होती हैं। फिर हस्ताक्षर के बाद फ़ील्ड को संशोधित करें और पुष्टि करें कि सत्यापन विफल होता है। यह आपको यह आंतरिक करने के लिए मजबूर करता है कि मानक एन्कोडिंग के हर बाइट से हस्ताक्षर में कैसे योगदान होता है।

**महत्वपूर्ण सीमा।** रसीदें तीन चीजें साबित करती हैं और केवल तीन चीजें: अभिज्ञान (इस कुंजी ने इस सामग्री पर हस्ताक्षर किया), अखंडता (सामग्री हस्ताक्षर के बाद से अपरिवर्तित है), और अनुक्रमण (यह रसीद उस रसीद के बाद आई)। वे यह साबित नहीं करतीं कि एजेंट की कार्रवाई सही थी, कि `policy_id` में नामित नीति वास्तव में मूल्यांकन की गई, या कि एजेंट ने हर नियम का पालन किया। रसीदें एक आधार हैं। शासन उस प्रणाली को कहते हैं जिसे आप उसके ऊपर बनाते हैं।

उस सीमा को ध्यान में रखते हुए पुनः पाठ पाठशाला README पढ़ें। टीमों द्वारा रसीदों के साथ की जाने वाली सबसे आम गलती यह मान लेना है कि "हमारे पास रसीदें हैं" का मतलब है "हम नियंत्रित हैं।" ऐसा नहीं है। रसीदें एजेंट के व्यवहार को ऑडिट योग्य बनाती हैं। वे इसे सही नहीं बनातीं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
